# <span style='color: pink'>**NEO4J GRAPH DATABASE**</span>

1. Connect to Neo4j

In [8]:
from neo4j import GraphDatabase
import pandas as pd

NEO4J_URI = "neo4j://127.0.0.1:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "tennis_db"   

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# quick connectivity test
with driver.session() as session:
    print(session.run("RETURN 'connected' AS status").single()["status"])


connected


2. Create constraints

In [9]:
constraints_cypher = [
    """
    CREATE CONSTRAINT player_key_unique IF NOT EXISTS
    FOR (p:Player)
    REQUIRE p.player_key IS UNIQUE
    """,
    """
    CREATE CONSTRAINT tournament_key_unique IF NOT EXISTS
    FOR (t:Tournament)
    REQUIRE t.tourney_key IS UNIQUE
    """,
    """
    CREATE CONSTRAINT match_id_unique IF NOT EXISTS
    FOR (m:Match)
    REQUIRE m.match_id IS UNIQUE
    """,
    """
    CREATE CONSTRAINT ranking_key_unique IF NOT EXISTS
    FOR (r:Ranking)
    REQUIRE r.ranking_key IS UNIQUE
    """
]

with driver.session() as session:
    for q in constraints_cypher:
        session.run(q)

print("Constraints created.")

Constraints created.


3. Helper: batch importer (UNWIND)

In [10]:
def run_in_batches(csv_path, query, batch_size=5000):
    """
    Reads a CSV and sends it to Neo4j in batches using UNWIND $rows.
    Works without using Neo4j import folder.
    """
    total = 0
    for chunk in pd.read_csv(csv_path, chunksize=batch_size, low_memory=False):
        rows = chunk.to_dict("records")
        with driver.session() as session:
            session.execute_write(lambda tx: tx.run(query, rows=rows))
        total += len(rows)
        if total % (batch_size * 10) == 0:
            print(f"Imported {total:,} rows from {csv_path}")
    print(f"Done {csv_path}: {total:,} rows")


4. Import Players

In [11]:
PLAYERS_CSV = "data/players_all.csv"

q_players = """
UNWIND $rows AS row
MERGE (p:Player {player_key: row.player_key})
SET
  p.player_id   = toInteger(row.player_id),
  p.player_name = row.player_name,
  p.ioc         = row.ioc,
  p.tour        = row.tour
"""
run_in_batches(PLAYERS_CSV, q_players, batch_size=5000)

Imported 50,000 rows from data/players_all.csv
Imported 100,000 rows from data/players_all.csv
Done data/players_all.csv: 135,977 rows


5. Import Tournaments

In [12]:
TOURNEYS_CSV = "data/tournaments_all.csv"

q_tourneys = """
UNWIND $rows AS row
MERGE (t:Tournament {tourney_key: row.tourney_key})
SET
  t.tourney_id    = row.tourney_id,
  t.tourney_name  = row.tourney_name,
  t.tourney_date  = date(row.tourney_date),
  t.tourney_level = row.tourney_level,
  t.surface       = row.surface,
  t.tour          = row.tour
"""

run_in_batches(TOURNEYS_CSV, q_tourneys, batch_size=2000)


Done data/tournaments_all.csv: 8,088 rows


6. Import Matches + Relationships (Match - Tournament and Match - Player)

In [13]:
MATCHES_CSV = "data/matches_all.csv"

q_matches_and_rels = """
UNWIND $rows AS row

// Match node
MERGE (m:Match {match_id: row.match_id})
SET
  m.round   = row.round,
  m.score   = row.score,
  m.best_of = toInteger(row.best_of),
  m.tour    = row.tour

// Relationships
WITH m, row
MATCH (t:Tournament {tourney_key: row.tourney_key})
MERGE (m)-[:BELONGS_TO]->(t)

WITH m, row
MATCH (w:Player {player_key: row.player_winner_key})
MERGE (m)-[:WINNER]->(w)

WITH m, row
MATCH (l:Player {player_key: row.player_loser_key})
MERGE (m)-[:LOSER]->(l)
"""

run_in_batches(MATCHES_CSV, q_matches_and_rels, batch_size=2000)


Imported 20,000 rows from data/matches_all.csv
Imported 40,000 rows from data/matches_all.csv
Imported 60,000 rows from data/matches_all.csv
Imported 80,000 rows from data/matches_all.csv
Imported 100,000 rows from data/matches_all.csv
Imported 120,000 rows from data/matches_all.csv
Imported 140,000 rows from data/matches_all.csv
Done data/matches_all.csv: 143,399 rows


7. Import Rankings + Relationship (Player - Ranking)

In [14]:
RANKINGS_CSV = "data/rankings_all.csv"

q_rankings_and_rels = """
UNWIND $rows AS row

MERGE (r:Ranking {ranking_key: row.ranking_key})
SET
  r.ranking_date = date(row.ranking_date),
  r.rank         = toInteger(row.rank),
  r.points       = toFloat(row.points),
  r.tour         = row.tour

WITH r, row
MATCH (p:Player {player_key: row.player_key})
MERGE (p)-[:HAS_RANKING]->(r)
"""

# rankings are huge -> smaller batch helps stability
run_in_batches(RANKINGS_CSV, q_rankings_and_rels, batch_size=5000)


Imported 50,000 rows from data/rankings_all.csv
Imported 100,000 rows from data/rankings_all.csv
Imported 150,000 rows from data/rankings_all.csv
Imported 200,000 rows from data/rankings_all.csv
Imported 250,000 rows from data/rankings_all.csv
Imported 300,000 rows from data/rankings_all.csv
Imported 350,000 rows from data/rankings_all.csv
Imported 400,000 rows from data/rankings_all.csv
Imported 450,000 rows from data/rankings_all.csv
Imported 500,000 rows from data/rankings_all.csv
Imported 550,000 rows from data/rankings_all.csv
Imported 600,000 rows from data/rankings_all.csv
Imported 650,000 rows from data/rankings_all.csv
Imported 700,000 rows from data/rankings_all.csv
Imported 750,000 rows from data/rankings_all.csv
Imported 800,000 rows from data/rankings_all.csv
Imported 850,000 rows from data/rankings_all.csv
Imported 900,000 rows from data/rankings_all.csv
Imported 950,000 rows from data/rankings_all.csv
Imported 1,000,000 rows from data/rankings_all.csv
Imported 1,050,000 

8. Sanity Check

In [15]:
q_checks = [
    ("players", "MATCH (p:Player) RETURN count(p) AS n"),
    ("tournaments", "MATCH (t:Tournament) RETURN count(t) AS n"),
    ("matches", "MATCH (m:Match) RETURN count(m) AS n"),
    ("rankings", "MATCH (r:Ranking) RETURN count(r) AS n"),
    ("BELONGS_TO", "MATCH (:Match)-[r:BELONGS_TO]->(:Tournament) RETURN count(r) AS n"),
    ("WINNER", "MATCH (:Match)-[r:WINNER]->(:Player) RETURN count(r) AS n"),
    ("LOSER", "MATCH (:Match)-[r:LOSER]->(:Player) RETURN count(r) AS n"),
    ("HAS_RANKING", "MATCH (:Player)-[r:HAS_RANKING]->(:Ranking) RETURN count(r) AS n"),
]

with driver.session() as session:
    for name, q in q_checks:
        n = session.run(q).single()["n"]
        print(f"{name:12s} : {n}")


players      : 136025
tournaments  : 8115
matches      : 286928
rankings     : 3804111
BELONGS_TO   : 286928
WINNER       : 286928
LOSER        : 286928
HAS_RANKING  : 3804111


# GDS

1. Connect to Neo4j desktop

In [16]:
from neo4j import GraphDatabase
import pandas as pd
import sys

#NEO4J_URI = "neo4j://127.0.0.1:7687"
#NEO4J_USER = "neo4j"
#NEO4J_PASSWORD = "tennis_db"   

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# quick connectivity test
with driver.session() as session:
    print(session.run("RETURN 'connected' AS status").single()["status"])

connected


2. Helper functions to run Cypher and get DataFrames

In [17]:
from typing import Optional, Dict
NEO4J_DATABASE = "neo4j"  

def run_cypher(query: str, 
               params: Optional[Dict] = None, 
               database: Optional[str] = None):
    
    params = params or {}
    database = database or NEO4J_DATABASE
    
    with driver.session(database=database) as session:
        result = session.run(query, params)
        return [record.data() for record in result]


def cypher_df(query: str, 
              params: Optional[Dict] = None, 
              database: Optional[str] = None) -> pd.DataFrame:
    
    rows = run_cypher(query, params=params, database=database)
    return pd.DataFrame(rows)

3. Running GDS projections from the notebook

In [18]:
# 1. Drop graph if it already exists (safe)
drop_atp = "CALL gds.graph.drop('atp_net', false) YIELD graphName;"
cypher_df(drop_atp)

drop_wta = "CALL gds.graph.drop('wta_net', false) YIELD graphName;"
cypher_df(drop_wta)

,graphName
0,wta_net


In [19]:
# 2.1 Create ATP projected graph (edit filter: tour or player_key)
create_atp = """
CALL gds.graph.project.cypher(
  'atp_net',
  'MATCH (p:Player) WHERE p.player_key STARTS WITH "ATP_" RETURN id(p) AS id',
  '
  MATCH (w:Player)<-[:WINNER]-(m:Match)-[:LOSER]->(l:Player)
  WHERE w.player_key STARTS WITH "ATP_" AND l.player_key STARTS WITH "ATP_"
  RETURN id(w) AS source, id(l) AS target
  '
)
YIELD graphName, nodeCount, relationshipCount
RETURN graphName, nodeCount, relationshipCount;
"""
cypher_df(create_atp)

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('gds.graph.project.cypher' has been replaced by 'gds.graph.project Cypher projection as an aggregation function')} {position: line: 2, column: 1, offset: 1} for query: '\nCALL gds.graph.project.cypher(\n  \'atp_net\',\n  \'MATCH (p:Player) WHERE p.player_key STARTS WITH "ATP_" RETURN id(p) AS id\',\n  \'\n  MATCH (w:Player)<-[:WINNER]-(m:Match)-[:LOSER]->(l:Player)\n  WHERE w.player_key STARTS WITH "ATP_" AND l.player_key STARTS WITH "ATP_"\n  RETURN id(w) AS source, id(l) AS target\n  \'\n)\nYIELD graphName, nodeCount, relationshipCount\nRETURN graphName, nodeCount, relationshipCount;\n'


,graphName,nodeCount,relationshipCount
0,atp_net,65989,149757


In [20]:
# 2.2 Create WTA projected graph (edit filter: tour or player_key)
create_wta = """
CALL gds.graph.project.cypher(
  'wta_net',
  'MATCH (p:Player) WHERE p.player_key STARTS WITH "WTA_" RETURN id(p) AS id',
  '
  MATCH (w:Player)<-[:WINNER]-(m:Match)-[:LOSER]->(l:Player)
  WHERE w.player_key STARTS WITH "WTA_" AND l.player_key STARTS WITH "WTA_"
  RETURN id(w) AS source, id(l) AS target
  '
)
YIELD graphName, nodeCount, relationshipCount
RETURN graphName, nodeCount, relationshipCount;
"""
cypher_df(create_wta)

Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.FeatureDeprecationWarning} {category: DEPRECATION} {title: This feature is deprecated and will be removed in future versions.} {description: The query used a deprecated procedure. ('gds.graph.project.cypher' has been replaced by 'gds.graph.project Cypher projection as an aggregation function')} {position: line: 2, column: 1, offset: 1} for query: '\nCALL gds.graph.project.cypher(\n  \'wta_net\',\n  \'MATCH (p:Player) WHERE p.player_key STARTS WITH "WTA_" RETURN id(p) AS id\',\n  \'\n  MATCH (w:Player)<-[:WINNER]-(m:Match)-[:LOSER]->(l:Player)\n  WHERE w.player_key STARTS WITH "WTA_" AND l.player_key STARTS WITH "WTA_"\n  RETURN id(w) AS source, id(l) AS target\n  \'\n)\nYIELD graphName, nodeCount, relationshipCount\nRETURN graphName, nodeCount, relationshipCount;\n'


,graphName,nodeCount,relationshipCount
0,wta_net,70036,137171


In [21]:
cypher_df("""
CALL gds.graph.list()
YIELD graphName, nodeCount, relationshipCount
RETURN graphName, nodeCount, relationshipCount
ORDER BY graphName;
""")

,graphName,nodeCount,relationshipCount
0,atp_net,65989,149757
1,wta_net,70036,137171


Perfect — those counts already tell you something meaningful about ATP vs WTA structure (even before running any algorithms):
- **ATP**: 65,989 nodes, 74,905 directed edges
- **WTA**: 70,036 nodes, 68,624 directed edges

So compared to WTA, ATP has fewer players but more distinct winner→loser connections.

That typically implies (we’ll verify with metrics):
- higher connectivity / interaction diversity in ATP, or
- more repeated cross-play across a broader set of opponents, or
- simply a structural artifact of your time span / data coverage (we can check)